(13)=
# Chapter 13: Symbolic Mathematics with SymPy

**Topics Covered:**
- What symbolic math is and why it matters in ChE
- Defining symbols and building expressions with SymPy
- Simplification, expansion, and substitution
- Symbolic calculus: differentiation, integration, and limits
- Solving algebraic and differential equations
- ChE application: van der Waals equation of state and Gibbs energy
- `lambdify`: converting symbolic expressions to fast numerical functions

In [ ]:
# ── All imports ──────────────────────────────────────────────────────────────
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing(use_unicode=True)   # pretty-print in Jupyter

(13.1)=
## 13.1 Motivation: Symbolic vs Numerical Math

In previous chapters, NumPy worked with **numbers** — floating-point approximations computed at runtime. SymPy works with **symbols** — exact mathematical objects that follow algebraic rules.

| | NumPy (numerical) | SymPy (symbolic) |
|---|---|---|
| Stores | Floating-point numbers | Exact expressions |
| `sqrt(2)` | `1.4142135...` | $\sqrt{2}$ (exact) |
| Derivative of $x^3$ | Not directly | $3x^2$ (exact) |
| Integral of $e^x$ | Numerical only | $e^x + C$ (exact) |
| Best for | Large arrays, speed | Derivations, exact answers |

**When do ChE engineers need symbolic math?**
- Deriving analytical expressions for $\frac{dP}{dT}$ or $\frac{dC_p}{dT}$ from an equation of state
- Integrating $C_p(T)$ exactly rather than numerically
- Solving the van der Waals equation for molar volume
- Finding steady-state solutions to ODEs before simulating them
- Verifying that a numerical answer has the right units and limiting behavior

The key workflow: **derive symbolically → convert to a fast numerical function with `lambdify` → evaluate or plot**.

(13.2)=
## 13.2 Symbols, Expressions, and Basic Operations

### 13.2.1 Defining Symbols

Everything in SymPy starts with `sp.symbols()`. A symbol is a named mathematical variable — it carries no numeric value until you assign one.

```python
x = sp.symbols('x')           # single symbol
x, y, z = sp.symbols('x y z') # multiple at once
T, P, V  = sp.symbols('T P V', positive=True)  # with assumptions
```

**Assumptions** tell SymPy about the mathematical properties of a symbol and allow it to simplify expressions correctly (e.g., $\sqrt{x^2} = x$ only if $x \geq 0$).

| Assumption | Effect |
|------------|--------|
| `positive=True` | $x > 0$; simplifies $\sqrt{x^2} = x$ |
| `real=True` | $x \in \mathbb{R}$; avoids complex branches |
| `integer=True` | $x \in \mathbb{Z}$; helps simplify floor/mod |

### 13.2.2 Building Expressions

Once you have symbols, you build expressions with ordinary Python operators. SymPy overloads `+`, `-`, `*`, `/`, `**` to produce symbolic objects instead of numbers.

In [1]:
import sympy as sp
sp.init_printing(use_unicode=True)

# ── Define symbols ────────────────────────────────────────────────────────────
x, y = sp.symbols('x y')
a, b, c = sp.symbols('a b c')

# ── Build expressions ─────────────────────────────────────────────────────────
expr1 = x**2 + 2*x + 1
expr2 = sp.sin(x) * sp.exp(-x)
expr3 = (a*x**2 + b*x + c)

print("expr1 =", expr1)
print("expr2 =", expr2)
print("expr3 =", expr3)

# ── Key operations ────────────────────────────────────────────────────────────
print("\n── expand ──")
print(sp.expand((x + 1)**3))          # multiply out

print("\n── factor ──")
print(sp.factor(x**3 - x**2 - x + 1)) # factor into irreducibles

print("\n── simplify ──")
print(sp.simplify((x**2 - 1) / (x - 1)))   # cancel common factor

print("\n── substitute (subs) ──")
print(expr1.subs(x, 3))               # evaluate at x = 3  → integer
print(expr1.subs(x, sp.pi))           # substitute symbolic value

print("\n── substitute multiple values ──")
print(expr3.subs([(a, 1), (b, -3), (c, 2)]))   # plug in all at once

expr1 = x**2 + 2*x + 1
expr2 = exp(-x)*sin(x)
expr3 = a*x**2 + b*x + c

── expand ──
x**3 + 3*x**2 + 3*x + 1

── factor ──
(x - 1)**2*(x + 1)

── simplify ──
x + 1

── substitute (subs) ──
16
1 + 2*pi + pi**2

── substitute multiple values ──
x**2 - 3*x + 2


### 13.2.3 Exact Arithmetic

SymPy never rounds. Fractions stay as fractions, $\pi$ stays as $\pi$, and $\sqrt{2}$ stays as $\sqrt{2}$ until you explicitly ask for a decimal with `sp.N()` or `.evalf()`.

In [2]:
# ── Exact arithmetic ─────────────────────────────────────────────────────────
print("Python float  :", 1/3 + 1/6)          # 0.5 (rounded)
print("SymPy rational:", sp.Rational(1,3) + sp.Rational(1,6))  # 1/2 (exact)

print("\nsqrt(8) =", sp.sqrt(8))              # 2√2  (exact)
print("pi      =", sp.pi)                     # π
print("pi (50 digits):", sp.N(sp.pi, 50))     # numerical to 50 sig figs

print("\nsp.E =", sp.E)                        # Euler's number, exact
print("exp(1) numerical:", sp.N(sp.E, 15))    # 2.71828182845905

Python float  : 0.5
SymPy rational: 1/2

sqrt(8) = 2*sqrt(2)
pi      = pi
pi (50 digits): 3.1415926535897932384626433832795028841971693993751

sp.E = E
exp(1) numerical: 2.71828182845905


(13.3)=
## 13.3 Symbolic Calculus

### 13.3.1 Differentiation

`sp.diff(expr, var)` differentiates `expr` with respect to `var`. Pass an integer as a third argument for higher-order derivatives.

$$
\frac{d}{dx}\left[x^n\right] = n x^{n-1}, \qquad
\frac{d^n}{dx^n}\left[e^x\right] = e^x
$$

In [ ]:
x = sp.symbols('x')

# ── First derivatives ─────────────────────────────────────────────────────────
print("d/dx [x^4]          =", sp.diff(x**4, x))
print("d/dx [sin(x)]       =", sp.diff(sp.sin(x), x))
print("d/dx [exp(-x^2)]    =", sp.diff(sp.exp(-x**2), x))
print("d/dx [ln(x)]        =", sp.diff(sp.ln(x), x))
print("d/dx [x^2 * sin(x)] =", sp.diff(x**2 * sp.sin(x), x))   # product rule

# ── Higher-order derivatives ──────────────────────────────────────────────────
print("\nd²/dx² [sin(x)] =", sp.diff(sp.sin(x), x, 2))
print("d³/dx³ [x^5]    =", sp.diff(x**5, x, 3))

# ── Partial derivatives ───────────────────────────────────────────────────────
T, V = sp.symbols('T V', positive=True)
R, a_vdw, b_vdw = sp.symbols('R a b', positive=True)

# Ideal gas pressure: P = RT/V
P_ideal = R * T / V
print("\n∂P/∂T (ideal gas) =", sp.diff(P_ideal, T))
print("∂P/∂V (ideal gas) =", sp.diff(P_ideal, V))

### 13.3.2 Integration

`sp.integrate(expr, var)` returns the **indefinite integral** (no constant of integration is shown but it is implied). For a **definite integral**, pass a tuple `(var, lower, upper)`.

$$
\int x^n \, dx = \frac{x^{n+1}}{n+1} + C, \qquad
\int_a^b f(x)\, dx = F(b) - F(a)
$$

In [ ]:
x, T = sp.symbols('x T', positive=True)

# ── Indefinite integrals ──────────────────────────────────────────────────────
print("∫ x^3 dx        =", sp.integrate(x**3, x))
print("∫ sin(x) dx     =", sp.integrate(sp.sin(x), x))
print("∫ exp(-x) dx    =", sp.integrate(sp.exp(-x), x))
print("∫ 1/x dx        =", sp.integrate(1/x, x))

# ── Definite integrals ────────────────────────────────────────────────────────
print("\n∫₀^π sin(x) dx =", sp.integrate(sp.sin(x), (x, 0, sp.pi)))   # → 2
print("∫₀^∞ exp(-x) dx =", sp.integrate(sp.exp(-x), (x, 0, sp.oo)))  # → 1

# ── ChE application: exact ΔH from a Cp polynomial fit ───────────────────────
# Cp(T) ≈ 20.71 + 0.06745 T - 4.498e-5 T² + 1.119e-8 T³  (CO2, J/mol/K)
Cp = 20.71 + 0.06745*T - 4.498e-5*T**2 + 1.119e-8*T**3

dH = sp.integrate(Cp, (T, 400, 900))
print(f"\nΔH = ∫₄₀₀⁹⁰⁰ Cp dT = {float(dH):.2f} J/mol  =  {float(dH)/1000:.4f} kJ/mol")

### 13.3.3 Limits

`sp.limit(expr, var, point)` evaluates the limit as `var → point`. Use `sp.oo` for $\infty$ and pass `'+'` or `'-'` as a fourth argument for one-sided limits.

In [ ]:
x = sp.symbols('x')

# ── Classic limits ────────────────────────────────────────────────────────────
print("lim x→0  sin(x)/x      =", sp.limit(sp.sin(x)/x, x, 0))         # → 1
print("lim x→∞  (1 + 1/x)^x  =", sp.limit((1 + 1/x)**x, x, sp.oo))    # → e
print("lim x→0+ ln(x)         =", sp.limit(sp.ln(x), x, 0, '+'))        # → -∞
print("lim x→∞  exp(-x)       =", sp.limit(sp.exp(-x), x, sp.oo))       # → 0

# ── ChE application: limiting ideal-gas behavior of van der Waals ─────────────
# P = RT/(V-b) - a/V²  as V→∞ should recover P → RT/V  (ideal gas)
R_s, T_s, V_s, a_s, b_s = sp.symbols('R T V a b', positive=True)
P_vdw = R_s*T_s/(V_s - b_s) - a_s/V_s**2

P_limit = sp.limit(P_vdw * V_s / (R_s * T_s), V_s, sp.oo)
print("\nlim V→∞  PV/(RT) for van der Waals =", P_limit, "  (→ 1 = ideal gas ✓)")

(13.4)=
## 13.4 Solving Equations

### 13.4.1 Algebraic Equations with `sp.solve`

`sp.solve(equation, variable)` returns all symbolic solutions to `equation = 0`. Pass an `sp.Eq(lhs, rhs)` object or just the expression (SymPy assumes it equals zero).

| Call | Meaning |
|------|---------|
| `sp.solve(expr, x)` | Solve `expr = 0` for `x` |
| `sp.solve(sp.Eq(lhs, rhs), x)` | Solve `lhs = rhs` for `x` |
| `sp.solve([eq1, eq2], [x, y])` | Solve a system of equations |

In [ ]:
x, y = sp.symbols('x y')
a, b, c = sp.symbols('a b c')

# ── Quadratic formula ─────────────────────────────────────────────────────────
print("Roots of ax² + bx + c = 0:")
roots = sp.solve(a*x**2 + b*x + c, x)
for r in roots:
    print(" ", r)

# ── Specific numerical example ────────────────────────────────────────────────
print("\nRoots of x² - 5x + 6 = 0:", sp.solve(x**2 - 5*x + 6, x))

# ── System of equations ───────────────────────────────────────────────────────
# 2x + 3y = 7
# x  - y  = 1
eq1 = sp.Eq(2*x + 3*y, 7)
eq2 = sp.Eq(x - y, 1)
sol = sp.solve([eq1, eq2], [x, y])
print("\nSystem solution:", sol)

# ── ChE: solve ideal gas for V ────────────────────────────────────────────────
P, V, R_s, T_s, n = sp.symbols('P V R T n', positive=True)
ideal_gas = sp.Eq(P*V, n*R_s*T_s)
V_sol = sp.solve(ideal_gas, V)
print("\nIdeal gas V =", V_sol[0])

### 13.4.2 Ordinary Differential Equations with `sp.dsolve`

`sp.dsolve(ode, func)` solves ordinary differential equations symbolically. You define the unknown function with `sp.Function` and write the ODE using `func(x).diff(x)`.

This is useful for:
- First-order decay / growth models (radioactive decay, first-order reactions)
- Finding the analytical steady-state of a CSTR or batch reactor
- Verifying numerical ODE solutions against exact answers

In [ ]:
t, k = sp.symbols('t k', positive=True)
C = sp.Function('C')    # C is the unknown function of t

# ── First-order reaction: dC/dt = -k*C ───────────────────────────────────────
ode1 = sp.Eq(C(t).diff(t), -k * C(t))
sol1 = sp.dsolve(ode1, C(t))
print("dC/dt = -kC  →  ", sol1)

# Apply initial condition C(0) = C0
C0 = sp.Symbol('C0', positive=True)
C1_const = sp.solve(sol1.rhs.subs(t, 0) - C0, sp.Symbol('C1'))[0]
C_exact = sol1.rhs.subs(sp.Symbol('C1'), C1_const)
print("With C(0)=C0  →  C(t) =", C_exact)

# ── Second-order ODE: d²y/dx² + y = 0  (harmonic oscillator) ─────────────────
x = sp.symbols('x')
y = sp.Function('y')
ode2 = sp.Eq(y(x).diff(x, 2) + y(x), 0)
sol2 = sp.dsolve(ode2, y(x))
print("\nd²y/dx² + y = 0  →  ", sol2)

# ── Energy balance ODE: dT/dt = (Q - UA(T-Tc)) / (m*Cp) ──────────────────────
# Simplified: dT/dt = alpha - beta*T  (first-order linear)
T_f = sp.Function('T')
alpha, beta = sp.symbols('alpha beta', positive=True)
ode3 = sp.Eq(T_f(t).diff(t), alpha - beta*T_f(t))
sol3 = sp.dsolve(ode3, T_f(t))
print("\ndT/dt = α - βT  →  ", sol3)
print("  (steady state T_ss = α/β as t→∞ ✓)")

(13.5)=
## 13.5 ChE Application: van der Waals Equation of State

The **van der Waals equation** corrects the ideal gas law for molecular volume ($b$) and intermolecular attractions ($a$):

$$
\left(P + \frac{a}{V^2}\right)(V - b) = RT
\qquad \Longleftrightarrow \qquad
P = \frac{RT}{V - b} - \frac{a}{V^2}
$$

This is a **cubic** equation in $V$ — it has up to three real roots, corresponding to liquid, two-phase, and vapor states. SymPy can:
1. Rearrange it into cubic form
2. Solve analytically for $V$
3. Compute $\left(\frac{\partial P}{\partial V}\right)_T$ for spinodal and critical-point analysis
4. Find the critical point where $\frac{\partial P}{\partial V} = \frac{\partial^2 P}{\partial V^2} = 0$

In [ ]:
P, V, T_s, R_s, a_s, b_s = sp.symbols('P V T R a b', positive=True)

# ── van der Waals pressure ────────────────────────────────────────────────────
P_vdw = R_s*T_s / (V - b_s) - a_s / V**2
print("P(V) =", P_vdw)

# ── dP/dV at constant T (compressibility / spinodal condition) ────────────────
dPdV = sp.diff(P_vdw, V)
dPdV_simplified = sp.simplify(dPdV)
print("\ndP/dV =", dPdV_simplified)

# ── d²P/dV² ──────────────────────────────────────────────────────────────────
d2PdV2 = sp.diff(P_vdw, V, 2)
print("\nd²P/dV² =", sp.simplify(d2PdV2))

# ── Critical point: solve dP/dV = 0 and d²P/dV² = 0 simultaneously ───────────
# Unknowns: Vc, Tc  (treating a, b, R as parameters)
V_c, T_c = sp.symbols('V_c T_c', positive=True)

crit_eq1 = dPdV.subs([(V, V_c), (T_s, T_c)])
crit_eq2 = d2PdV2.subs([(V, V_c), (T_s, T_c)])

crit_sol = sp.solve([crit_eq1, crit_eq2], [V_c, T_c])
print("\nCritical point:")
print("  V_c =", crit_sol[V_c])
print("  T_c =", crit_sol[T_c])

# Critical pressure
P_c = P_vdw.subs([(V, crit_sol[V_c]), (T_s, crit_sol[T_c])])
P_c_simplified = sp.simplify(P_c)
print("  P_c =", P_c_simplified)

### 13.5.1 Plotting the van der Waals P–V Isotherm

Using the symbolic expression and `lambdify`, we can plot isotherms for CO$_2$ ($a = 0.3658$ J·m³/mol², $b = 4.286\times10^{-5}$ m³/mol) and visualize the phase transition region.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# CO2 van der Waals constants
a_co2 = 0.3658       # J·m³/mol²
b_co2 = 4.286e-5     # m³/mol
R_val = 8.314        # J/(mol·K)
T_c_co2 = 8 * a_co2 / (27 * R_val * b_co2)   # ≈ 304 K  (critical temp)

# Convert symbolic P_vdw → fast numerical function
P_num = sp.lambdify((V, T_s, R_s, a_s, b_s), P_vdw, 'numpy')

V_arr = np.linspace(1.1*b_co2, 6e-4, 2000)   # m³/mol

fig, ax = plt.subplots(figsize=(9, 5))
for T_val, color in zip([250, 280, T_c_co2, 330, 380],
                        ['tab:blue', 'tab:cyan', 'tab:red', 'tab:orange', 'tab:gray']):
    P_arr = P_num(V_arr, T_val, R_val, a_co2, b_co2) / 1e6   # Pa → MPa
    label = f'T = {T_val:.0f} K' + (' (critical)' if abs(T_val - T_c_co2) < 1 else '')
    lw = 2.5 if abs(T_val - T_c_co2) < 1 else 1.8
    ax.plot(V_arr * 1e6, P_arr, color=color, linewidth=lw, label=label)  # V in cm³/mol

ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xlim(0, 600); ax.set_ylim(-2, 15)
ax.set_xlabel('Molar volume $V$ (cm³/mol)')
ax.set_ylabel('Pressure $P$ (MPa)')
ax.set_title('van der Waals isotherms for CO$_2$\n(negative pressure region is unphysical — Maxwell construction needed)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"CO₂ critical temperature: T_c = {T_c_co2:.1f} K  ({T_c_co2-273.15:.1f} °C)")
print(f"CO₂ critical pressure:    P_c = {a_co2/(27*b_co2**2)/1e6:.2f} MPa")

(13.6)=
## 13.6 `lambdify`: Bridging SymPy and NumPy

SymPy expressions are **symbolic objects** — they cannot be evaluated on NumPy arrays directly. `sp.lambdify` converts a symbolic expression into a regular Python function backed by NumPy (or any other numerical library), giving you the best of both worlds: symbolic derivation + numerical speed.

```python
f_num = sp.lambdify(variables, expression, 'numpy')
```

| Argument | Description |
|----------|-------------|
| `variables` | Symbol or tuple of symbols — become function arguments |
| `expression` | Any SymPy expression |
| `'numpy'` | Backend: `'numpy'` (default), `'scipy'`, `'math'`, or a dict |

**Typical workflow:**
1. Define symbols and derive the expression symbolically (e.g., take a derivative, integrate, simplify)
2. Call `lambdify` to get a fast numerical function
3. Evaluate on arrays, plot, or pass to a solver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

x = sp.symbols('x')

# ── Step 1: derive symbolically ───────────────────────────────────────────────
f_sym   = sp.sin(x) * sp.exp(-x / 3)
df_sym  = sp.diff(f_sym, x)
d2f_sym = sp.diff(f_sym, x, 2)

print("f(x)   =", f_sym)
print("f'(x)  =", df_sym)
print("f''(x) =", d2f_sym)

# ── Step 2: convert to numerical functions ────────────────────────────────────
f   = sp.lambdify(x, f_sym,   'numpy')
df  = sp.lambdify(x, df_sym,  'numpy')
d2f = sp.lambdify(x, d2f_sym, 'numpy')

# ── Step 3: evaluate and plot ─────────────────────────────────────────────────
x_arr = np.linspace(0, 4*np.pi, 400)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_arr, f(x_arr),   'k-',  linewidth=2,   label=r'$f(x) = \sin(x)\,e^{-x/3}$')
ax.plot(x_arr, df(x_arr),  'b--', linewidth=1.8, label=r"$f'(x)$")
ax.plot(x_arr, d2f(x_arr), 'r:',  linewidth=1.8, label=r"$f''(x)$")
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Symbolic differentiation → lambdify → numerical plot')
ax.legend()
plt.tight_layout()
plt.show()

# ── Find zeros of f'(x) numerically using the lambdified derivative ───────────
from scipy.optimize import brentq
zeros = []
for i in range(len(x_arr) - 1):
    if df(x_arr[i]) * df(x_arr[i+1]) < 0:   # sign change → root
        root = brentq(df, x_arr[i], x_arr[i+1])
        zeros.append(root)

print("\nCritical points of f(x) on [0, 4π]:")
for z in zeros:
    print(f"  x = {z:.4f},  f(x) = {f(z):.4f},  f'(x) ≈ {df(z):.2e}")

(13.7)=
## 13.7 Summary

| Concept | SymPy tool | Example |
|---------|-----------|---------|
| **Define symbols** | `sp.symbols('x y', positive=True)` | Variables with assumptions |
| **Exact arithmetic** | `sp.Rational`, `sp.pi`, `sp.E` | $\frac{1}{3} + \frac{1}{6} = \frac{1}{2}$ exactly |
| **Expand** | `sp.expand(expr)` | $(x+1)^3 \to x^3 + 3x^2 + 3x + 1$ |
| **Factor** | `sp.factor(expr)` | $x^2 - 1 \to (x-1)(x+1)$ |
| **Simplify** | `sp.simplify(expr)` | Cancel, trig identities, etc. |
| **Substitute** | `expr.subs(x, val)` | Plug in a value or another symbol |
| **Differentiate** | `sp.diff(expr, x, n)` | $n$-th derivative w.r.t. $x$ |
| **Integrate** | `sp.integrate(expr, x)` | Indefinite; `(x, a, b)` for definite |
| **Limit** | `sp.limit(expr, x, pt)` | $\lim_{x\to 0}\frac{\sin x}{x} = 1$ |
| **Solve algebraic** | `sp.solve(eq, x)` | Roots, quadratic formula, systems |
| **Solve ODE** | `sp.dsolve(ode, f(x))` | Analytical solution with constant $C_1$ |
| **Numerical eval** | `expr.evalf(n)` or `sp.N(expr, n)` | $\pi$ to 50 digits |
| **Vectorize** | `sp.lambdify(x, expr, 'numpy')` | SymPy → NumPy-compatible function |

**Golden rule:** Use SymPy to *derive* — exact formulas, derivatives, integrals, and solutions. Use NumPy/SciPy to *compute* — fast array operations, plotting, and numerical solvers. Connect the two with `lambdify`.

**ChE use cases covered:**
- $\partial P / \partial T$ and $\partial P / \partial V$ from the ideal gas and van der Waals EOS
- Exact $\Delta H = \int C_p\, dT$ from a polynomial fit
- Critical point $(V_c, T_c, P_c)$ of van der Waals gas from $\nabla P = 0$
- First-order reaction $dC/dt = -kC$ solved analytically
- Reactor energy balance ODE solved to find steady-state temperature